# Model Comparison with `django-pgai`

Step by step walkthrough of the four embedding models declared on `pgai_example.models.Movie`, using only the `django-pgai` ORM surface.

Run this notebook from the project root, inside an environment where the Django app and its dependencies are installed (the simplest path is the `django` container in `docker-compose.yml`).

Prerequisites:
- The four vectorizers must already have written embeddings. Check with `./manage.sh pgai list`.
- The `Movie` table must be populated. Run `./setup.sh build` if it isn't.

## 1. Bootstrap Django

A notebook is not a Django management command, so configure the settings module and call `django.setup()` before importing any models.

In [ ]:
import os
import django

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
django.setup()

from pgai_example.models import Movie
from django_pgai.db.semantic_search.expressions import semantic_score
from django.db.models import F

## 2. The four vectorizers under test

Each vectorizer is exposed as a field on `Movie`. They all read the same `overview` column (the three with `source_field="overview"` are *proxy* vectorizers) and produce a different embedding.

In [ ]:
VARIANTS = [
    ("overview",            "qwen3-embedding",       4096),
    ("overview_mxbai",      "mxbai-embed-large",     1024),
    ("overview_minilm",     "all-MiniLM-L6-v2",       384),
    ("overview_snowflake",  "snowflake-arctic-embed", 1024),
]

print(f"Movies in DB: {Movie.objects.count():,}")
for field, model, dim in VARIANTS:
    print(f"  {field:22s} -> {model:24s} ({dim} dim)")

## 3. One model, one query: `similar_in.<field>.find()`

The friendliest entry point. Returns a list of `SemanticResult` objects, each exposing `.score`, `.relevance`, `.match_count`, and `.instance`.

In [ ]:
QUERY = "samurai revenge"

results = Movie.similar_in.overview_snowflake.find(QUERY, limit=5)
for r in results:
    print(f"  {r.score:.3f}  {r.instance.title}")

## 4. Side by side: same query, all four models

The interesting part of the comparison. The same `find()` call is repeated against every vectorizer field, and the results are tabulated for read across.

In [ ]:
def top_n(field: str, query: str, limit: int = 5):
    accessor = getattr(Movie.similar_in, field)
    return accessor.find(query, limit=limit)

per_model = {field: top_n(field, QUERY, limit=5) for field, *_ in VARIANTS}

header = "  ".join(f"{m[:18]:18s}" for _, m, _ in VARIANTS)
print(f"#  {header}")
print("-" * (3 + len(header)))
for i in range(5):
    row_cells = []
    for field, *_ in VARIANTS:
        hits = per_model[field]
        if i < len(hits):
            cell = f"{hits[i].instance.title[:13]:13s} {hits[i].score:.3f}"
        else:
            cell = "-"
        row_cells.append(f"{cell:18s}")
    print(f"{i+1}  " + "  ".join(row_cells))

## 5. Score distributions: top 1 per model

Absolute similarity scores are not comparable across models. Each embedding family lives in its own score range. Picking thresholds without checking this is a common mistake.

Run the same query across all four models and inspect where each one's top 1 lands.

In [ ]:
import matplotlib.pyplot as plt

labels = [m for _, m, _ in VARIANTS]
top1_scores = [per_model[field][0].score if per_model[field] else 0.0 for field, *_ in VARIANTS]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels, top1_scores)
ax.set_ylim(0, 1)
ax.set_ylabel("top 1 similarity score")
ax.set_title(f'Top 1 score per model for "{QUERY}"')
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 6. Latency per model

Run the same query a few times against each model, time the `find()` call, and report the mean. This measures the full path: embed the query, send to Postgres, run the pgvector similarity scan, return rows.

In [ ]:
import statistics, time

RUNS = 5

def time_find(field: str, query: str, runs: int = RUNS):
    accessor = getattr(Movie.similar_in, field)
    # warm up once
    accessor.find(query, limit=5)
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        accessor.find(query, limit=5)
        samples.append((time.perf_counter() - t0) * 1000.0)
    return statistics.fmean(samples)

means = [time_find(field, QUERY) for field, *_ in VARIANTS]
for (field, model, _), ms in zip(VARIANTS, means):
    print(f"  {model:24s} {ms:7.1f} ms mean ({field})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels, means)
ax.set_ylabel("mean latency (ms)")
ax.set_title(f'End to end query latency per model for "{QUERY}"')
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 7. Tuning lever: threshold cutoff

Passing `threshold=` drops results below a cutoff. The cutoff is per model. A useful threshold on one embedding is noise on another. Sweep a few values on one model to see the gap between relevant and irrelevant hits.

In [ ]:
FIELD = "overview_snowflake"
accessor = getattr(Movie.similar_in, FIELD)

for t in (0.55, 0.70, 0.74, 0.80):
    hits = accessor.find(QUERY, threshold=t, limit=10)
    print(f"threshold = {t:.2f}  ->  {len(hits)} hits")
    for h in hits[:3]:
        print(f"    {h.score:.3f}  {h.instance.title}")

## 8. Tuning lever: ranking strategy (`rank_by`)

When a document chunks into multiple pieces, multiple chunks can match a query. `rank_by=` decides how those per chunk scores collapse into one document score.

- `best`: highest single chunk similarity (default)
- `relevance`: best score weighted by number of matching chunks
- `count`: number of chunks above the threshold

In [ ]:
for strategy in ("best", "relevance", "count"):
    print(f"\nrank_by = {strategy!r}")
    hits = accessor.find(QUERY, rank_by=strategy, limit=5)
    for h in hits:
        print(f"    {h.score:.3f}  match_count={h.match_count}  {h.instance.title}")

## 9. Tuning lever: blending multiple models

When two models disagree on the top hit, a blended ranker rewards documents that score well under *both* models. This is a stronger relevance signal than either model alone.

Two ways to express the blend:

**A.** Pass a list of fields to `semantic_rank()` and let the plugin combine internally.

In [ ]:
blended_qs = Movie.objects.semantic_rank(
    QUERY,
    fields=["overview_snowflake", "overview_mxbai"],
)[:5]

for m in blended_qs:
    print(f"  {m.semantic_score:.3f}  {m.title}")

**B.** Annotate each model's score separately, then combine with arithmetic. Use this when you want explicit weights or want to mix in non semantic signals (recency, popularity, an editorial boost).

In [ ]:
weighted_qs = (
    Movie.objects
    .annotate(
        s_snow=semantic_score("overview_snowflake", QUERY),
        s_mxbai=semantic_score("overview_mxbai", QUERY),
    )
    .annotate(blended=0.6 * F("s_snow") + 0.4 * F("s_mxbai"))
    .order_by("-blended")[:5]
)

for m in weighted_qs:
    print(f"  blended={m.blended:.3f}  snow={m.s_snow:.3f}  mxbai={m.s_mxbai:.3f}  {m.title}")

## 10. Structured filter plus semantic rank

Compose any standard Django `.filter()` with `semantic_score()`. The structured predicate narrows the candidate set. The semantic score re ranks the survivors. This is the workhorse pattern for production search where "find something relevant and matching these constraints" is the real requirement.

In [ ]:
action_qs = (
    Movie.objects
    .filter(genres__icontains="Action")
    .annotate(score=semantic_score("overview_snowflake", QUERY))
    .order_by("-score")[:5]
)

for m in action_qs:
    print(f"  {m.score:.3f}  {m.title}  [{m.genres}]")

## 11. Try your own query

Change `QUERY` below, re run cells 4 onwards, and see how the model rankings shift. Queries that exercise different semantic shapes (literal phrasing, mood, specific named entities, multilingual) tend to expose the biggest disagreements between models.

In [ ]:
QUERY = "existential dread"  # try: "cooking mice", "hacker breaks into the pentagon", or your own